Authors: Malcolm Zartman & Evan Petersen

Sharable Link: https://colab.research.google.com/drive/1DKSOpSaAxZEKNZ28DTNQvngbfFkU5jOF?usp=sharing


Briefly descirbe the dataset:
- What are the inputs?
  - Inputs:
    - 50,000 training images
    - 10,000 testing images
    - Each image has 32x32 pixels and 3 channels for rgb
- What are the labels?
  - There are 10 labels [0-9], each label is associated with a category ranging from vehicles to animals.
- What are the dimensions of this dataset?
  - x_train: (50000,32,32,3)
  - y_train: (50000,1)
  - x_test: (10000,32,32,3)
  - y_test: (10000,1)

In [ ]:
from keras.datasets import cifar10
import numpy as np

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print(x_train.shape, "\n", y_train.shape, "\n", x_test.shape, "\n", y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
(50000, 32, 32, 3) 
 (50000, 1) 
 (10000, 32, 32, 3) 
 (10000, 1)


In [ ]:
# reshaping to a flattened array, which is what PyTorch expects for their labels
y_train = np.squeeze(y_train)
y_test = np.squeeze(y_test)

# reordering since PyTorch expects channel first images
x_train = x_train.transpose(0,3,1,2)
x_test = x_test.transpose(0,3,1,2)

# converting to float to turn the rgb channels to a range of 0-1 for a more efficient model
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

print(x_train.shape, "\n", y_train.shape, "\n", x_test.shape, "\n", y_test.shape)

(50000, 3, 32, 32) 
 (50000,) 
 (10000, 3, 32, 32) 
 (10000,)


In [ ]:
import torch

x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [ ]:
import torch.nn as nn

class SimpleMLP(nn.Module):
  def __init__(self, activation, hidden_neurons=64, extra_layer=False):
    super().__init__()

    layers = []

    # input -> hidden
    layers.append(nn.Flatten())
    layers.append(nn.Linear(3*32*32, hidden_neurons))
    layers.append(activation)

    # optional extra hidden layer
    if extra_layer:
      layers.append(nn.Linear(hidden_neurons, hidden_neurons))
      layers.append(activation)

    # hidden -> output logit
    layers.append(nn.Linear(hidden_neurons, 10))

    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)

In [ ]:
activations = {
    "ReLU": nn.ReLU(),        # Using this b/c this ReLU performs really well, not only in terms of speed but also accuracy
    "Sigmoid": nn.Sigmoid(),  # Using sigmoid b/c I want to see the differences between good & bad activation functions for this problem (this should show the saturation effect heavily)
    "Tanh": nn.Tanh()         # Using tanh b/c I want this is a bit smoother than sigmoid but will also perform poorly due to saturation
}

for name, act in activations.items():
    print(f"\nTraining model with {name}")

    model = SimpleMLP(activation=act)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(20):
        optimizer.zero_grad()
        logits = model(x_train)
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        logits = model(x_test)
        preds = logits.argmax(dim=1)
        accuracy = (preds == y_test).float().mean().item()

    print(f"{name} accuracy: {accuracy:.4f}")


Training model with ReLU
ReLU accuracy: 0.2577

Training model with Sigmoid
Sigmoid accuracy: 0.2928

Training model with Tanh
Tanh accuracy: 0.3096


These results were not necessarily expected but also expected. I did think that ReLU would've performed better due to my previous knowledge on how it works and performs, but due to the instructions of this assignment and how shallow and simple this model is, ReLU is actually not the best option and is rather slightly outpreformed by the more stable sigmoid and tanh activation functions.

In [ ]:

print(f"\nTraining model with Tanh")

model = SimpleMLP(activation=nn.Tanh(), hidden_neurons=128, extra_layer=True)
criterion = nn.MultiMarginLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(20):
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    logits = model(x_test)
    preds = logits.argmax(dim=1)
    accuracy = (preds == y_test).float().mean().item()

print(f"Tanh accuracy: {accuracy:.4f}")


Training model with Tanh
Tanh accuracy: 0.1104


After changing the number of neurons, adding an additional layer, having a different loss function & optimizer, that led to a substantial decrease in accuracy. I think the biggest reason for this is because of the optimizer and of course the simplicity of the network. So, in the next run through I will change back to the Adam optimizer as it requires less fine tuning and does better with smaller networks.

In [ ]:

print(f"\nTraining model with Tanh")

model = SimpleMLP(activation=nn.Tanh(), hidden_neurons=128, extra_layer=True)
criterion = nn.MultiMarginLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    logits = model(x_test)
    preds = logits.argmax(dim=1)
    accuracy = (preds == y_test).float().mean().item()

print(f"Tanh accuracy: {accuracy:.4f}")


Training model with Tanh
Tanh accuracy: 0.3301


This simple change in optimizer type led to a 3x (+22%) in accuracy performance as predicted.

In [ ]:

print(f"\nTraining model with Tanh")

model = SimpleMLP(activation=nn.Tanh(), hidden_neurons=256, extra_layer=True)
criterion = nn.MultiMarginLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    logits = model(x_test)
    preds = logits.argmax(dim=1)
    accuracy = (preds == y_test).float().mean().item()

print(f"Tanh accuracy: {accuracy:.4f}")


Training model with Tanh
Tanh accuracy: 0.3334


Increase in neurons led to a marginal increase in accuracy. Checking to see if the loss function will lead to any change in accuracy.

In [ ]:

print(f"\nTraining model with Tanh")

model = SimpleMLP(activation=nn.Tanh(), hidden_neurons=256, extra_layer=True)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    logits = model(x_test)
    preds = logits.argmax(dim=1)
    accuracy = (preds == y_test).float().mean().item()

print(f"Tanh accuracy: {accuracy:.4f}")


Training model with Tanh
Tanh accuracy: 0.3336


The swapping of loss functions led to essentially no change in accuracy at this stage. Which is surprising as cross entropy usually lends greater results as compared to to multi margin loss, but since the classes can be easily separable and the simplicity of the model, this is probably the case for the same results.

In [ ]:

print(f"\nTraining model with Tanh")

model = SimpleMLP(activation=nn.Tanh(), hidden_neurons=512, extra_layer=True)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    logits = model(x_test)
    preds = logits.argmax(dim=1)
    accuracy = (preds == y_test).float().mean().item()

print(f"Tanh accuracy: {accuracy:.4f}")


Training model with Tanh
Tanh accuracy: 0.3088


The additional increase in neurons led to a decrease in accuracy. This is likely because of how simple the model and network is. At this point an architectural change is more needed than any additional fine tuning (obviously).

But from the results, the best combination of hyperparameters for this multi-class classification problem are: the Tanh activation function due to its stable bounded nature, 256 neurons in the hidden layers, an additional hidden layer, a cross entropy loss function, and the Adam optimizer with a learning rate of 0.001.